# Exercise 5.1: Signal + background fit at 125 GeV

From *Programming in High Energy Particle Physics*, Chapter 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch05/ex05_1_solution.ipynb)

Exercise 5.1 Signal + background fit at 125 GeV Generate a toy dataset with signal (Gaussian at 125 GeV, \(\sigma = 2\) GeV) plus background (exponential). Fit with a signal+background model and extract the signal yield and its uncertainty. Compare binned and unbinned likelihood results.

<details><summary>Hint</summary>

Adapt Listing 5.4 : change the mass range to, say, 100–160 GeV and the mean to 125 GeV. For the binned fit, create a RooDataHist from the dataset ( data-&gt;binnedClone() ) and fit it with the same model. With fine enough bins the two yields and uncertainties should agree closely; they start to differ when the bin width becomes comparable to \(\sigma\).

</details>

### Colab setup

The first run takes about 3–5 minutes and restarts the runtime once. After restart, run all cells from the top again. The next cell installs ROOT from conda-forge only if it is missing.

In [ ]:
import sys, importlib.util
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and importlib.util.find_spec("ROOT") is None:
    %pip install -q condacolab
    import condacolab
    condacolab.install()

In [ ]:
import importlib.util
if IN_COLAB and importlib.util.find_spec("ROOT") is None:
    !mamba install -q -y -c conda-forge root
import ROOT
ROOT.gROOT.SetBatch(True)
print("ROOT",ROOT.gROOT.GetVersion())

### Generate and fit

The model and variable names follow Listing 5.4. The default 0.5 GeV bins are narrower than the 2 GeV signal resolution.

In [ ]:
%%writefile ex05_1.C
#include <RooAddPdf.h>
#include <RooDataHist.h>
#include <RooDataSet.h>
#include <RooExponential.h>
#include <RooFitResult.h>
#include <RooGaussian.h>
#include <RooPlot.h>
#include <RooRandom.h>
#include <RooRealVar.h>
#include <TCanvas.h>
#include <TStyle.h>
#include <memory>
#include <iostream>
void ex05_1() {
  using namespace RooFit;
  RooRandom::randomGenerator()->SetSeed(42);
  RooRealVar mass("mass","m_{ll}",100,160,"GeV");
  mass.setBins(120); // 0.5 GeV, much narrower than sigma=2 GeV
  RooRealVar mean("mean","peak",125,122,128,"GeV");
  RooRealVar sigma("sigma","resolution",2,1,4,"GeV");
  RooGaussian signal("signal","signal",mass,mean,sigma);
  RooRealVar lambda("lambda","background slope",-0.035,-0.1,-0.001);
  RooExponential background("background","background",mass,lambda);
  RooRealVar nsig("nsig","signal yield",2000,0,6000);
  RooRealVar nbkg("nbkg","background yield",5000,0,10000);
  RooAddPdf model("model","signal + background",RooArgList(signal,background),RooArgList(nsig,nbkg));
  std::unique_ptr<RooDataSet> data{model.generate(mass,Extended())};
  nsig.setVal(1800); nbkg.setVal(5200);
  std::unique_ptr<RooFitResult> unbinned{model.fitTo(*data,Save(),Extended(true),PrintLevel(-1))};
  const double uYield=nsig.getVal(), uErr=nsig.getError();
  std::cout << "unbinned status=" << unbinned->status() << " covQual=" << unbinned->covQual()
            << " nsig=" << uYield << " +/- " << uErr << std::endl;
  std::unique_ptr<RooDataHist> binned{data->binnedClone()};
  nsig.setVal(1800); nbkg.setVal(5200);
  std::unique_ptr<RooFitResult> bfit{model.fitTo(*binned,Save(),Extended(true),PrintLevel(-1))};
  std::cout << "binned status=" << bfit->status() << " covQual=" << bfit->covQual()
            << " nsig=" << nsig.getVal() << " +/- " << nsig.getError() << std::endl;
  auto frame=mass.frame();
  binned->plotOn(frame);
  model.plotOn(frame);
  model.plotOn(frame,Components("background"),LineStyle(kDashed),LineColor(kRed));
  frame->GetXaxis()->SetTitle("m_{ll} [GeV]");
  frame->GetYaxis()->SetTitle("Events / 0.5 GeV");
  TCanvas c("c","signal plus background",700,500);
  frame->Draw(); c.SaveAs("ex05_1_fit.pdf");
}


In [ ]:
!root -l -b -q ex05_1.C

With fine binning, binned and unbinned signal yields and errors should agree within toy fluctuations. Widen the bins to explore loss of shape information.